# Stage 1 & 2: Problem Scenario, Dataset Identification & Sourcing Pipeline
**Scenario:** Urban Traffic Incident Severity Prediction & Emergency Dispatch Optimization
**Stakeholders:** 911 Computer-Aided Dispatch (CAD) Operators, Municipal DOT Traffic Centers
**Prediction Target:** `Severity` (Scale 1 to 4: Minor delay to Arterial gridlock)
**Mining Task:** Multiclass Classification

### Real-World Operational Logic:
Emergency dispatchers must allocate first responders within seconds of an incoming call. 
Post-incident metrics (such as queue length or clearance duration) do not exist yet. 
This pipeline loads tabular atmospheric, roadway, and temporal variables to classify incident impact immediately.

### Dataset Source & Citation:
- **Source:** US Accidents (Countrywide Traffic Accident Dataset, 2016–2023), Sobhan Moosavi et al., published via Kaggle.
- **Academic Citation:** Moosavi, S., Samavatian, M. H., Parthasarathy, S., & Ramnath, R. (2019). *A Countrywide Traffic Accident Dataset*. arXiv:1906.05409.
- **Scale:** 7.7 Million records reduced to a representative 1.0 Million sample via chunk streaming.

1. Problem Formulation & Sourcing

In [1]:
import os
import pandas as pd
import numpy as np

# 21 Required features (Filtering out 25 post-incident and high-cardinality leakage columns)
selected_columns = [
    "Severity", "Start_Time", "Temperature(F)", "Humidity(%)", 
    "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)", 
    "Weather_Condition", "Sunrise_Sunset", "State", "Amenity", "Bump", 
    "Crossing", "Give_Way", "Junction", "No_Exit", "Railway", 
    "Roundabout", "Stop", "Traffic_Signal"
]

print(f"Target Feature Selection: {len(selected_columns)} operational attributes selected.")

Target Feature Selection: 21 operational attributes selected.


2. Chunk-Streaming Ingestion & Schema Filter

In [2]:
import os
import pandas as pd
import numpy as np

# 21 Required features (Filtering out 25 post-incident and high-cardinality leakage columns)
selected_columns = [
    "Severity", "Start_Time", "Temperature(F)", "Humidity(%)", 
    "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)", 
    "Weather_Condition", "Sunrise_Sunset", "State", "Amenity", "Bump", 
    "Crossing", "Give_Way", "Junction", "No_Exit", "Railway", 
    "Roundabout", "Stop", "Traffic_Signal"
]

print(f"Target Feature Selection: {len(selected_columns)} operational attributes selected.")

Target Feature Selection: 21 operational attributes selected.


3. 1M Sampling Validation & Master Dataset Check

In [3]:
# Master dataset loading check
file_path = "../data/US_Accidents_2016_2023_1M.csv"

# Load initial inspection records
df_sample = pd.read_csv(file_path, nrows=10_000)

print("=== DATASET EXTRACTION AUDIT ===")
print(f"Target Working File: {file_path}")
print(f"Features Loaded:     {df_sample.shape[1]}")
print(f"Verified Columns:    {list(df_sample.columns)}")
df_sample.head(3)

=== DATASET EXTRACTION AUDIT ===
Target Working File: ../data/US_Accidents_2016_2023_1M.csv
Features Loaded:     21
Verified Columns:    ['Severity', 'Start_Time', 'State', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset']


,Severity,Start_Time,State,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in),Weather_Condition,...,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Stop,Traffic_Signal,Sunrise_Sunset
0,4,1/14/2016 20:18,PA,31.0,69.0,29.53,10.0,3.0,0.00,Fair,...,False,False,False,False,False,False,False,False,False,Night
1,2,2/8/2016 6:15,OH,36.0,97.0,29.70,10.0,NaN,0.02,Overcast,...,False,False,False,True,False,False,False,False,False,Night
2,2,2/8/2016 11:51,OH,33.1,92.0,29.63,0.5,3.5,0.08,Snow,...,False,False,False,True,False,False,False,False,False,Day


4. Class Representation Verification across Sample

In [4]:
# Check severity representation in master data
severity_dist = df_sample['Severity'].value_counts(normalize=True).sort_index() * 100
print("--- Class Distribution in Ingested Sample ---")
for sev, pct in severity_dist.items():
    print(f"Severity Tier {sev}: {pct:.2f}%")

--- Class Distribution in Ingested Sample ---
Severity Tier 1: 0.02%
Severity Tier 2: 63.86%
Severity Tier 3: 32.59%
Severity Tier 4: 3.53%
